# 07 — Personalized Retention Recommendations

## Purpose

This notebook converts the outputs from the customer-value and segmentation layers into actionable retention recommendations.

Inputs:

```text
customer_features.csv
        +
customer_segments.csv
        +
customer_clv_predictions.csv
        +
customer_shap_values.csv
```

Output:

```text
Customer
   ↓
Value + Recency + Frequency + Segment + Model Drivers
   ↓
Retention Risk / Opportunity
   ↓
Personalized Recommendation
```

This is a **business-rule recommendation layer**, not another machine-learning model. The rules are transparent, auditable, and easy to expose through the API later.

## Business questions

- Which customers should receive retention attention first?
- Which high-value customers are becoming inactive?
- Which customers are healthy and should receive loyalty actions?
- Which customers have low value and low activity?
- What action should the business take for each customer?
- Why was that action recommended?

### Design principle

A recommendation should contain both:

1. **Action** — what the business should do.
2. **Reason** — the customer behavior that triggered the action.

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import plotly.express as px

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_PATH = PROCESSED_DIR / "retention_recommendations.csv"

FEATURE_PATH = PROCESSED_DIR / "customer_features.csv"
SEGMENT_PATH = PROCESSED_DIR / "customer_segments.csv"
CLV_PATH = PROCESSED_DIR / "customer_clv_predictions.csv"
SHAP_PATH = PROCESSED_DIR / "customer_shap_values.csv"

print(f"Project root: {PROJECT_ROOT}")

## 1. Load required artifacts

In [ ]:
required_paths = [
    FEATURE_PATH,
    SEGMENT_PATH,
    CLV_PATH,
    SHAP_PATH,
]

missing_paths = [
    path for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Missing required artifacts:\n"
        + "\n".join(str(path) for path in missing_paths)
        + "\nRun the preceding notebooks first."
    )

features = pd.read_csv(FEATURE_PATH)
segments = pd.read_csv(SEGMENT_PATH)
clv = pd.read_csv(CLV_PATH)
shap_values = pd.read_csv(SHAP_PATH)

print("Customer features:", features.shape)
print("Segments:", segments.shape)
print("CLV predictions:", clv.shape)
print("SHAP values:", shap_values.shape)

## 2. Select and merge customer-level information

The recommendation engine needs one row per customer.

We retain the most relevant fields:

- recency
- order frequency
- historical revenue
- predicted future revenue
- customer segment
- RFM score
- review behavior
- delivery behavior

In [ ]:
feature_columns = [
    "customer_unique_id",
    "recency_days",
    "order_count",
    "total_revenue",
    "average_order_value",
    "total_items",
    "average_review_score",
    "negative_review_rate",
    "average_delivery_days",
    "average_delivery_vs_estimate_days",
    "rfm_score",
    "customer_segment",
]

feature_columns = [
    column
    for column in feature_columns
    if column in features.columns
]

base = features[feature_columns].copy()

clv_columns = [
    "customer_unique_id",
    "predicted_future_revenue",
]

clv_columns = [
    column
    for column in clv_columns
    if column in clv.columns
]

base = base.merge(
    clv[clv_columns],
    on="customer_unique_id",
    how="left",
)

segment_columns = [
    "customer_unique_id",
    "cluster_id",
    "segment",
]

segment_columns = [
    column
    for column in segment_columns
    if column in segments.columns
]

base = base.merge(
    segments[segment_columns],
    on="customer_unique_id",
    how="left",
)

base = base.drop_duplicates(
    subset=["customer_unique_id"]
).reset_index(drop=True)

print("Merged customer table:", base.shape)
display(base.head())

## 3. Create business thresholds

We use percentile-based thresholds instead of arbitrary currency values.

This makes the rules adapt to the actual distribution of the dataset.

Thresholds include:

- high predicted value
- low predicted value
- high recency
- recent activity
- high frequency
- low review quality

In [ ]:
base["predicted_future_revenue"] = pd.to_numeric(
    base["predicted_future_revenue"],
    errors="coerce",
).fillna(0)

base["recency_days"] = pd.to_numeric(
    base["recency_days"],
    errors="coerce",
).fillna(0)

base["order_count"] = pd.to_numeric(
    base["order_count"],
    errors="coerce",
).fillna(0)

base["total_revenue"] = pd.to_numeric(
    base["total_revenue"],
    errors="coerce",
).fillna(0)

base["average_review_score"] = pd.to_numeric(
    base["average_review_score"],
    errors="coerce",
).fillna(0)

high_value_threshold = base[
    "predicted_future_revenue"
].quantile(0.75)

low_value_threshold = base[
    "predicted_future_revenue"
].quantile(0.25)

high_recency_threshold = base[
    "recency_days"
].quantile(0.75)

recent_activity_threshold = base[
    "recency_days"
].quantile(0.25)

high_frequency_threshold = base[
    "order_count"
].quantile(0.75)

low_review_threshold = base[
    "average_review_score"
].replace(0, np.nan).quantile(0.25)

thresholds = pd.DataFrame({
    "threshold": [
        "high_value",
        "low_value",
        "high_recency",
        "recent_activity",
        "high_frequency",
        "low_review",
    ],
    "value": [
        high_value_threshold,
        low_value_threshold,
        high_recency_threshold,
        recent_activity_threshold,
        high_frequency_threshold,
        low_review_threshold,
    ],
})

display(thresholds.round(2))

## 4. Assign retention priority

Priority logic:

### Critical
High predicted future value + high recency.

### High
High predicted value with either inactivity or another warning signal.

### Medium
Moderate value or moderate engagement that may benefit from targeted action.

### Low
Low predicted value and no major risk signal.

The goal is to prioritize limited retention resources.

In [ ]:
def assign_priority(row):
    value = row["predicted_future_revenue"]
    recency = row["recency_days"]
    review = row["average_review_score"]

    if (
        value >= high_value_threshold
        and recency >= high_recency_threshold
    ):
        return "Critical"

    if (
        value >= high_value_threshold
        and (
            recency >= high_recency_threshold
            or (
                review > 0
                and review <= low_review_threshold
            )
        )
    ):
        return "High"

    if (
        value >= low_value_threshold
        or recency <= recent_activity_threshold
    ):
        return "Medium"

    return "Low"


base["retention_priority"] = base.apply(
    assign_priority,
    axis=1,
)

priority_summary = (
    base["retention_priority"]
    .value_counts()
    .reindex(
        ["Critical", "High", "Medium", "Low"],
        fill_value=0,
    )
    .rename_axis("priority")
    .reset_index(name="customers")
)

priority_summary["percentage"] = (
    100
    * priority_summary["customers"]
    / len(base)
)

display(priority_summary.round(2))

fig = px.bar(
    priority_summary,
    x="priority",
    y="customers",
    title="Retention Priority Distribution",
    labels={
        "priority": "Priority",
        "customers": "Customers",
    },
)
fig.show()

## 5. Generate transparent recommendation rules

The rule engine considers:

1. predicted future value,
2. customer recency,
3. purchase frequency,
4. review satisfaction,
5. segment,
6. delivery experience.

The first matching high-priority rule determines the recommendation.

This makes the system deterministic and auditable.

In [ ]:
def generate_recommendation(row):
    value = row["predicted_future_revenue"]
    recency = row["recency_days"]
    orders = row["order_count"]
    review = row["average_review_score"]

    if (
        value >= high_value_threshold
        and recency >= high_recency_threshold
    ):
        return (
            "Win-back campaign",
            "High predicted future value combined with "
            "long purchase inactivity.",
        )

    if (
        value >= high_value_threshold
        and review > 0
        and review <= low_review_threshold
    ):
        return (
            "Service recovery outreach",
            "High predicted value but weak review performance.",
        )

    if (
        value >= high_value_threshold
        and orders >= high_frequency_threshold
    ):
        return (
            "VIP loyalty offer",
            "High predicted value and strong purchase frequency.",
        )

    if recency <= recent_activity_threshold:
        return (
            "Cross-sell recommendation",
            "Customer is recently active and is a good candidate "
            "for additional product recommendations.",
        )

    if orders <= 1 and value <= low_value_threshold:
        return (
            "First-to-second-purchase campaign",
            "Customer has limited purchase history and low predicted value.",
        )

    if value <= low_value_threshold:
        return (
            "Low-cost engagement",
            "Predicted future value is relatively low, "
            "so retention spend should remain limited.",
        )

    return (
        "Standard retention nurture",
        "Customer shows moderate value and engagement.",
    )


recommendation_pairs = base.apply(
    generate_recommendation,
    axis=1,
    result_type="expand",
)

recommendation_pairs.columns = [
    "recommended_action",
    "recommendation_reason",
]

base = pd.concat(
    [base, recommendation_pairs],
    axis=1,
)

display(
    base[
        [
            "customer_unique_id",
            "predicted_future_revenue",
            "recency_days",
            "retention_priority",
            "recommended_action",
            "recommendation_reason",
        ]
    ].head(20)
)

## 6. Add recommended retention strategy

The action is translated into a practical strategy.

This is intentionally generic enough to avoid assuming a specific marketing platform or discount budget.

In [ ]:
strategy_map = {
    "Win-back campaign": (
        "Personalized reactivation message with a relevant product "
        "recommendation and time-limited incentive."
    ),
    "Service recovery outreach": (
        "Contact the customer, identify service friction, and "
        "offer recovery support before promotional selling."
    ),
    "VIP loyalty offer": (
        "Prioritize loyalty benefits, early access, bundles, or "
        "exclusive product recommendations."
    ),
    "Cross-sell recommendation": (
        "Recommend complementary products based on purchase history."
    ),
    "First-to-second-purchase campaign": (
        "Use a low-cost follow-up campaign focused on converting "
        "the first purchase into a repeat purchase."
    ),
    "Low-cost engagement": (
        "Use inexpensive personalized content rather than "
        "high-cost incentives."
    ),
    "Standard retention nurture": (
        "Maintain relevant product recommendations and periodic "
        "customer engagement."
    ),
}

base["retention_strategy"] = base[
    "recommended_action"
].map(strategy_map)

display(
    base[
        [
            "customer_unique_id",
            "retention_priority",
            "recommended_action",
            "retention_strategy",
        ]
    ].head(20)
)

## 7. Create an opportunity score

The opportunity score prioritizes customers using:

- predicted future value,
- recency risk,
- purchase frequency.

It is not a probability of churn.

It is a business prioritization score.

In [ ]:
value_percentile = (
    base["predicted_future_revenue"]
    .rank(pct=True)
)

recency_risk = (
    base["recency_days"]
    .rank(pct=True)
)

frequency_strength = (
    base["order_count"]
    .rank(pct=True)
)

base["opportunity_score"] = (
    0.50 * value_percentile
    + 0.35 * recency_risk
    + 0.15 * (1 - frequency_strength)
)

base["opportunity_score"] = (
    100 * base["opportunity_score"]
)

base = base.sort_values(
    "opportunity_score",
    ascending=False,
).reset_index(drop=True)

display(
    base[
        [
            "customer_unique_id",
            "predicted_future_revenue",
            "recency_days",
            "order_count",
            "opportunity_score",
            "retention_priority",
            "recommended_action",
        ]
    ].head(20)
)

## 8. Priority-level business summary

In [ ]:
priority_profile = (
    base.groupby("retention_priority")
    .agg(
        customers=("customer_unique_id", "count"),
        average_predicted_value=(
            "predicted_future_revenue",
            "mean",
        ),
        total_predicted_value=(
            "predicted_future_revenue",
            "sum",
        ),
        average_recency_days=(
            "recency_days",
            "mean",
        ),
        average_orders=("order_count", "mean"),
        average_opportunity_score=(
            "opportunity_score",
            "mean",
        ),
    )
    .reindex(["Critical", "High", "Medium", "Low"])
)

display(priority_profile.round(2))

## 9. Recommended actions by priority

In [ ]:
action_summary = (
    base.groupby(
        [
            "retention_priority",
            "recommended_action",
        ]
    )
    .size()
    .reset_index(name="customers")
    .sort_values(
        ["retention_priority", "customers"],
        ascending=[True, False],
    )
)

display(action_summary)

## 10. Save the recommendation dataset

The resulting table is the bridge between analytics and the application layer.

Later:

```text
FastAPI
   ↓
RecommendationService
   ↓
retention_recommendations.csv
   ↓
Streamlit
```

The production service can eventually replace the CSV lookup with database queries while preserving the same business logic.

In [ ]:
OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

base.to_csv(
    OUTPUT_PATH,
    index=False,
)

print(f"Saved retention recommendations: {OUTPUT_PATH}")
print(f"Rows: {len(base):,}")
print(f"Columns: {len(base.columns):,}")

# Final validation

Expected artifact:

```text
data/processed/retention_recommendations.csv
```

Each customer should have:

```text
customer_unique_id
predicted_future_revenue
retention_priority
recommended_action
recommendation_reason
retention_strategy
opportunity_score
```

### Important interpretation

These recommendations are **decision-support rules**.

They are not claims that a customer will churn, and they do not establish causal effects.

The next layer can combine these recommendations with SHAP explanations and Groq to produce a natural-language AI business insight.